<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/sessions/Session11_lstm_and_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۱۱: LSTM، GRU و Word Embedding برای دسته‌بندی متن

### اهداف این جلسه

- درک سازوکار دروازه‌ها (Gates) در LSTM و GRU و چرایی حل مشکل محوشدگی گرادیان
- پیاده‌سازی سلول LSTM از صفر و تطبیق آن با `nn.LSTMCell`
- آماده‌سازی متن: Tokenization، ساخت واژگان (Vocabulary)، Padding و Packing
- آشنایی با Word Embedding و لایه `nn.Embedding`
- تحلیل احساس نقدهای فیلم IMDB با LSTM دوطرفه و مقایسه با GRU و یک خط مبنای ساده

> **نکتهٔ اجرا:** این نوت‌بوک روی GPU بسیار سریع‌تر اجرا می‌شود. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. از RNN به LSTM

در جلسه قبل دیدیم RNN ساده نمی‌تواند اطلاعات را در دنباله‌های طولانی نگه دارد، چون در هر گام حالت پنهان **کاملاً بازنویسی** می‌شود و گرادیان از حاصل‌ضرب تعداد زیادی ماتریس عبور می‌کند.

**Long Short-Term Memory (LSTM)** (Hochreiter & Schmidhuber, 1997) دو تغییر کلیدی دارد:

۱. یک **حالت سلول (Cell State)** $\mathbf{c}_t$ جدا از حالت پنهان، که مثل یک «نوار نقاله» اطلاعات را در طول زمان با تغییرات **جمعی** (نه ضربی) منتقل می‌کند.

۲. سه **دروازه (Gate)** که با Sigmoid (خروجی بین ۰ و ۱) تصمیم می‌گیرند چه اطلاعاتی عبور کند:

$$\mathbf{f}_t = \sigma(W_f[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \quad \text{(Forget gate: چه چیزی از حافظه پاک شود)}$$

$$\mathbf{i}_t = \sigma(W_i[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i) \quad \text{(Input gate: چه مقدار از اطلاعات جدید نوشته شود)}$$

$$\tilde{\mathbf{c}}_t = \tanh(W_c[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c) \quad \text{(اطلاعات کاندید)}$$

$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \quad \text{(به‌روزرسانی حافظه)}$$

$$\mathbf{o}_t = \sigma(W_o[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o),\qquad \mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t) \quad \text{(Output gate)}$$

**چرا گرادیان محو نمی‌شود؟** مشتق $\mathbf{c}_t$ نسبت به $\mathbf{c}_{t-1}$ برابر $\mathbf{f}_t$ است. اگر شبکه یاد بگیرد Forget Gate را نزدیک ۱ نگه دارد، گرادیان بدون تضعیف از گام‌های زیادی عبور می‌کند؛ همان ایده «مسیر مستقیم» که در ResNet دیدیم.

### GRU

**Gated Recurrent Unit** (Cho et al., 2014) نسخه ساده‌شده LSTM است: حالت سلول و حالت پنهان یکی شده‌اند و فقط دو دروازه دارد (**Update** و **Reset**):

$$\mathbf{z}_t = \sigma(W_z[\mathbf{h}_{t-1}, \mathbf{x}_t]),\quad \mathbf{r}_t = \sigma(W_r[\mathbf{h}_{t-1}, \mathbf{x}_t]),\quad \tilde{\mathbf{h}}_t = \tanh(W[\mathbf{r}_t \odot \mathbf{h}_{t-1}, \mathbf{x}_t]),\quad \mathbf{h}_t = (1-\mathbf{z}_t)\odot\mathbf{h}_{t-1} + \mathbf{z}_t\odot\tilde{\mathbf{h}}_t$$

GRU حدود ۲۵٪ پارامتر کمتر از LSTM دارد و معمولاً عملکرد مشابهی دارد.

</div>

In [ ]:
import re
import time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
torch.manual_seed(42)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. پیاده‌سازی سلول LSTM از صفر

چهار تبدیل خطی (سه دروازه و اطلاعات کاندید) را در یک ماتریس بزرگ ترکیب می‌کنیم؛ PyTorch هم همین کار را می‌کند و ترتیب وزن‌ها در آن `input, forget, cell, output` است:

</div>

In [ ]:
class MyLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.x2g = nn.Linear(input_size, 4 * hidden_size)            # all four gates at once
        self.h2g = nn.Linear(hidden_size, 4 * hidden_size)

    def forward(self, x, state):
        h, c = state
        gates = self.x2g(x) + self.h2g(h)
        i, f, g, o = gates.chunk(4, dim=1)                           # PyTorch order: input, forget, cell, output
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g                                            # additive memory update
        h = o * torch.tanh(c)
        return h, c


ref = nn.LSTMCell(3, 4)
mine = MyLSTMCell(3, 4)
with torch.no_grad():
    mine.x2g.weight.copy_(ref.weight_ih); mine.x2g.bias.copy_(ref.bias_ih)
    mine.h2g.weight.copy_(ref.weight_hh); mine.h2g.bias.copy_(ref.bias_hh)
x, h0, c0 = torch.randn(2, 3), torch.zeros(2, 4), torch.zeros(2, 4)
h_ref, c_ref = ref(x, (h0, c0))
h_mine, c_mine = mine(x, (h0, c0))
print('h matches:', torch.allclose(h_ref, h_mine, atol=1e-6), '| c matches:', torch.allclose(c_ref, c_mine, atol=1e-6))

for name, cell in [('RNN', nn.RNN), ('GRU', nn.GRU), ('LSTM', nn.LSTM)]:
    print(f'{name:4s} parameters (input 100, hidden 128): {sum(p.numel() for p in cell(100, 128).parameters()):,}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. دیتاست IMDB

دیتاست **IMDB** (Maas et al., 2011) شامل ۵۰٬۰۰۰ نقد فیلم انگلیسی با برچسب مثبت یا منفی است: ۲۵٬۰۰۰ برای آموزش و ۲۵٬۰۰۰ برای آزمون. آن را از Hugging Face Hub با کتابخانه `datasets` بارگذاری می‌کنیم (در جلسه ۱۴ بیشتر با این کتابخانه کار می‌کنیم).

</div>

In [ ]:
%pip install -q datasets

In [ ]:
from datasets import load_dataset

imdb = load_dataset('stanfordnlp/imdb')
train_texts, train_labels = imdb['train']['text'], imdb['train']['label']
test_texts, test_labels = imdb['test']['text'], imdb['test']['label']
print('train:', len(train_texts), '| test:', len(test_texts), '| positive ratio:', np.mean(train_labels))
print('\nexample review:\n', train_texts[0][:400], '...\nlabel:', train_labels[0])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. از متن به تنسور

شبکه عصبی فقط عدد می‌فهمد. سه مرحله لازم است:

**۱. Tokenization:** شکستن متن به واحدهای کوچک‌تر (Token). اینجا از یک Tokenizer ساده در سطح کلمه با عبارت باقاعده (Regex) استفاده می‌کنیم. مدل‌های مدرن از Tokenizerهای زیرکلمه‌ای (Subword) مثل BPE و WordPiece استفاده می‌کنند (جلسه ۱۴).

**۲. واژگان (Vocabulary):** نگاشت هر Token به یک عدد صحیح. فقط پرتکرارترین کلمات **داده آموزشی** را نگه می‌داریم و دو Token ویژه اضافه می‌کنیم: `<pad>` (برای هم‌طول کردن دنباله‌ها) و `<unk>` (برای کلمات ناشناخته).

**۳. Padding:** دنباله‌های یک Batch طول‌های متفاوتی دارند؛ دنباله‌های کوتاه‌تر را با `<pad>` تا طول بلندترین دنباله پر می‌کنیم.

</div>

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")


def tokenize(text):
    text = text.lower().replace('<br />', ' ')                  # IMDB reviews contain HTML line breaks
    return TOKEN_RE.findall(text)


print(tokenize("This movie wasn't great, but the actors were AMAZING!<br />8/10"))

MAX_VOCAB, MAX_LEN = 20_000, 256
counter = Counter(tok for text in train_texts for tok in tokenize(text))
itos = ['<pad>', '<unk>'] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
stoi = {w: i for i, w in enumerate(itos)}
PAD, UNK = 0, 1
print(f'distinct tokens in training data: {len(counter):,} | vocabulary size: {len(itos):,}')

lengths = [len(tokenize(t)) for t in train_texts[:5000]]
print(f'review length: median {np.median(lengths):.0f} tokens, 90th percentile {np.percentile(lengths, 90):.0f}')

In [ ]:
def encode(text):
    ids = [stoi.get(tok, UNK) for tok in tokenize(text)]
    return ids[:MAX_LEN]                                          # truncate long reviews


class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.data = [torch.tensor(encode(t) or [UNK]) for t in texts]
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i], self.labels[i]


def collate(batch):
    """Pad variable-length sequences in a batch and keep their true lengths."""
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True, padding_value=PAD)
    return padded, lengths, torch.stack(labels)


train_ds, test_ds = ReviewDataset(train_texts, train_labels), ReviewDataset(test_texts, test_labels)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=256, collate_fn=collate)

X, L, y = next(iter(train_loader))
print('padded batch:', tuple(X.shape), '| lengths:', L[:8].tolist())
print('decoded start of first review:', ' '.join(itos[i] for i in X[0, :20].tolist()))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۵. Word Embedding

ساده‌ترین نمایش یک کلمه بردار One-hot با طول واژگان (۲۰٬۰۰۰) است: بسیار بزرگ، تُنُک و بدون هیچ مفهوم شباهتی (فاصله «خوب» تا «عالی» برابر فاصله «خوب» تا «میز» است).

**Word Embedding** هر کلمه را به یک **بردار چگال (Dense)** با ابعاد کم (مثلاً ۱۰۰ تا ۳۰۰) نگاشت می‌کند که **یاد گرفته می‌شود**. کلمات با معنی یا کاربرد مشابه بردارهای نزدیک به هم پیدا می‌کنند. لایه `nn.Embedding(vocab_size, dim)` در واقع یک جدول جست‌وجو (Lookup Table) است: ماتریسی با یک سطر برای هر کلمه که ورودی عدد صحیح را به سطر متناظر تبدیل می‌کند. این کار معادل ضرب One-hot در یک ماتریس وزن است، اما بسیار سریع‌تر.

Embeddingها می‌توانند همراه با مسئله آموزش ببینند (کاری که اینجا می‌کنیم) یا از مدل‌های ازپیش‌آموزش‌دیده مثل **Word2Vec** (Mikolov et al., 2013)، **GloVe** و **fastText** (که برای فارسی هم بردار دارد) بارگذاری شوند.

</div>

In [ ]:
emb = nn.Embedding(num_embeddings=len(itos), embedding_dim=100, padding_idx=PAD)
out = emb(X)
print('token ids:', tuple(X.shape), '-> embeddings:', tuple(out.shape))
print('the <pad> vector is fixed at zero:', emb.weight[PAD].abs().sum().item() == 0)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. مدل: Embedding، LSTM دوطرفه و طبقه‌بند

**Packing:** اگر دنباله Pad‌شده را مستقیم به LSTM بدهیم، LSTM روی Tokenهای `<pad>` هم گام برمی‌دارد و حالت پنهان آخر خراب می‌شود. `pack_padded_sequence` طول واقعی هر دنباله را به LSTM اعلام می‌کند تا فقط روی Tokenهای واقعی محاسبه کند و `h_n` حالت **آخرین Token واقعی** هر دنباله باشد.

برای LSTM دوطرفه، حالت پنهان نهایی دو جهت را کنار هم می‌گذاریم: `h_n[-2]` (آخرین لایه، جهت رفت) و `h_n[-1]` (آخرین لایه، جهت برگشت).

</div>

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128, cell='lstm', num_layers=1, bidirectional=True, dropout=0.3):
        super().__init__()
        self.cell = cell
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        rnn_cls = {'lstm': nn.LSTM, 'gru': nn.GRU}[cell]
        self.rnn = rnn_cls(emb_dim, hidden, num_layers=num_layers, batch_first=True, bidirectional=bidirectional)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), 2)
        self.bidirectional = bidirectional

    def forward(self, x, lengths):
        e = self.dropout(self.embedding(x))
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)
        h_n = h[0] if self.cell == 'lstm' else h                    # LSTM returns (h_n, c_n)
        last = torch.cat([h_n[-2], h_n[-1]], dim=1) if self.bidirectional else h_n[-1]
        return self.fc(self.dropout(last))


class BagOfEmbeddings(nn.Module):
    """Baseline: average the word embeddings and ignore word order."""

    def __init__(self, vocab_size, emb_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.fc = nn.Linear(emb_dim, 2)

    def forward(self, x, lengths):
        summed = self.embedding(x).sum(dim=1)
        return self.fc(summed / lengths.unsqueeze(1).to(summed.device))

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = correct = n = 0
    with torch.set_grad_enabled(training):
        for X, L, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X, L)
            loss = nn.functional.cross_entropy(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(y); correct += (logits.argmax(1) == y).sum().item(); n += len(y)
    return total_loss / n, correct / n


EPOCHS = 5
results = {}
for name, make in [
    ('bag of embeddings', lambda: BagOfEmbeddings(len(itos))),
    ('GRU (bidirectional)', lambda: RNNClassifier(len(itos), cell='gru')),
    ('LSTM (bidirectional)', lambda: RNNClassifier(len(itos), cell='lstm')),
]:
    torch.manual_seed(0)
    model = make().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    start, accs = time.time(), []
    for epoch in range(EPOCHS):
        run_epoch(model, train_loader, optimizer)
        accs.append(run_epoch(model, test_loader)[1])
    results[name] = (model, accs)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'{name:22s} params {n_params:>10,} | test acc per epoch {np.round(accs, 3)} | {time.time() - start:.0f}s')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مدل ساده میانگین Embedding (که ترتیب کلمات را نادیده می‌گیرد) خط مبنای بسیار قوی‌ای برای تحلیل احساس است، چون حضور کلماتی مثل *great* یا *awful* اطلاعات زیادی دارد. LSTM و GRU با در نظر گرفتن ترتیب کلمات، الگوهایی مثل نفی (*not good*) را بهتر مدل می‌کنند. در عمل همیشه یک خط مبنای ساده بسازید تا بدانید پیچیدگی اضافه چقدر ارزش دارد.

در جلسه ۱۴ خواهیم دید که مدل‌های Transformer ازپیش‌آموزش‌دیده (مثل BERT) با Fine-tuning روی همین داده به دقت بالای ۹۳٪ می‌رسند؛ قدرت آن‌ها از پیش‌آموزش روی متن عظیم و سازوکار Attention می‌آید.

### امتحان روی جمله‌های دلخواه

</div>

In [ ]:
@torch.no_grad()
def predict_sentiment(model, text):
    model.eval()
    ids = torch.tensor([encode(text) or [UNK]]).to(device)
    prob = torch.softmax(model(ids, torch.tensor([ids.shape[1]])), dim=1)[0, 1].item()
    return f'{"positive" if prob > 0.5 else "negative"} ({prob:.2f})'


lstm_model = results['LSTM (bidirectional)'][0]
bag_model = results['bag of embeddings'][0]
for s in ['An absolute masterpiece with brilliant acting.',
          'The plot was boring and the ending made no sense.',
          'I expected a great film, but it was not good at all.',
          'Not bad at all, I actually enjoyed it.']:
    print(f'{s:58s} LSTM: {predict_sentiment(lstm_model, s):18s} bag: {predict_sentiment(bag_model, s)}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### نگاهی به Embeddingهای یادگرفته‌شده

نزدیک‌ترین کلمات به چند کلمه، بر اساس **شباهت کسینوسی (Cosine Similarity)** بردارهای Embedding مدل LSTM. چون Embeddingها فقط برای تشخیص احساس آموزش دیده‌اند، «نزدیک» بیشتر به معنای «بار احساسی مشابه» است:

</div>

In [ ]:
E = nn.functional.normalize(lstm_model.embedding.weight.detach().cpu(), dim=1)
for word in ['excellent', 'terrible', 'boring', 'funny']:
    if word in stoi:
        sims = E @ E[stoi[word]]
        nearest = [itos[i] for i in sims.topk(8).indices.tolist() if itos[i] != word][:6]
        print(f'{word:10s} -> {", ".join(nearest)}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## جمع‌بندی

- LSTM با حالت سلول و سه دروازه (Forget، Input، Output) اطلاعات را با به‌روزرسانی **جمعی** در طول زمان حفظ می‌کند؛ گرادیان از مسیر $\mathbf{c}_t$ کمتر محو می‌شود.
- GRU نسخه ساده‌تر با دو دروازه و پارامتر کمتر است و عملکرد مشابهی دارد.
- خط لوله متن: Tokenization، ساخت واژگان (با `<pad>` و `<unk>`)، تبدیل به اعداد صحیح و Padding در `collate_fn`.
- `nn.Embedding` یک جدول جست‌وجوی قابل آموزش است که کلمات را به بردارهای چگال تبدیل می‌کند.
- `pack_padded_sequence` باعث می‌شود RNN روی Tokenهای Padding محاسبه نکند.
- برای طبقه‌بندی، حالت نهایی دو جهت LSTM دوطرفه را کنار هم می‌گذاریم.
- همیشه با یک خط مبنای ساده (مثل میانگین Embedding) مقایسه کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. `MyGRUCell` را از صفر پیاده‌سازی و با `nn.GRUCell` تطبیق دهید (ترتیب دروازه‌ها در PyTorch: reset، update، new).

۲. `MAX_LEN` را ۶۴ و ۵۱۲ بگیرید. دقت و زمان آموزش چه تغییری می‌کند؟

۳. به‌جای حالت نهایی، **Max Pooling روی زمان** (بیشینه خروجی‌های LSTM در همه گام‌ها) را به طبقه‌بند بدهید. برای این کار از `pad_packed_sequence` استفاده کنید و مراقب مکان‌های Padding باشید.

۴. Embeddingهای ازپیش‌آموزش‌دیده GloVe (۱۰۰ بعدی) را دانلود و در لایه Embedding بارگذاری کنید (`nn.Embedding.from_pretrained`). یک بار منجمد و یک بار قابل آموزش امتحان کنید.

۵. همین خط لوله را روی یک دیتاست احساس فارسی (مثلاً نظرات کاربران دیجی‌کالا یا اسنپ‌فود از Hugging Face Hub) اجرا کنید. چه تغییری در Tokenizer لازم است؟

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** `MyGRUCell` را از صفر پیاده‌سازی و با `nn.GRUCell` تطبیق دهید (ترتیب دروازه‌ها در PyTorch: reset، update، new).

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** `MAX_LEN` را ۶۴ و ۵۱۲ بگیرید. دقت و زمان آموزش چه تغییری می‌کند؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** به‌جای حالت نهایی، **Max Pooling روی زمان** (بیشینه خروجی‌های LSTM در همه گام‌ها) را به طبقه‌بند بدهید. برای این کار از `pad_packed_sequence` استفاده کنید و مراقب مکان‌های Padding باشید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** Embeddingهای ازپیش‌آموزش‌دیده GloVe (۱۰۰ بعدی) را دانلود و در لایه Embedding بارگذاری کنید (`nn.Embedding.from_pretrained`). یک بار منجمد و یک بار قابل آموزش امتحان کنید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۵.** همین خط لوله را روی یک دیتاست احساس فارسی (مثلاً نظرات کاربران دیجی‌کالا یا اسنپ‌فود از Hugging Face Hub) اجرا کنید. چه تغییری در Tokenizer لازم است؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید
